# BS49903 · DS/AI for Biology
## Assignment 3

**Due: Friday, December 4, 11:59 PM**

**Instructions.** This notebook is designed to run on **Google Colab**. Work through the exercises in order. Fill in every cell marked `#### YOUR CODE HERE ####` and answer the written questions in the markdown cells provided. Do not delete the provided cells.

**What to submit:**
1. This notebook (`.ipynb`) with all outputs visible.
2. The two Part 2 screen captures, `before.png` and `after.png`, uploaded as image files through the submission link.

Part 1 trains a small U-Net from scratch using provided ground-truth labels. Part 2 is a human-in-the-loop exercise using the Cellpose GUI on your own computer, in your local copy of the course GitHub repository.

-----
## Setup — Google Colab and Google Drive

Part 1 runs in Colab. It reads its data from, and saves its trained model to, the `assignment03` folder in your Google Drive, so they are kept after the Colab session ends. (Part 2 runs on your own computer.)

1. Copy the whole `assignment03` folder from the course GitHub repository into **My Drive** in Google Drive. In Google Drive, choose **New › Folder upload** and select the `assignment03` folder in your local copy of the repository. Do not rename the folder or move files inside it. You should then have:

   ```
   My Drive/assignment03/
   ├── assignment03.ipynb
   ├── part1/   train_images.tif, train_labels.tif
   └── part2/   dapi_cutout.tif
   ```
2. Open the notebook from there: in **My Drive › assignment03**, right-click `assignment03.ipynb` and choose **Open with › Google Colaboratory**.
3. Turn on a GPU: **Runtime › Change runtime type › T4 GPU › Save**. CPU also works, but training is slower.
4. Run the cell below. When asked, allow Colab to access your Google Drive.

If Colab disconnects, run this cell and the imports cell again before you continue.

In [ ]:
# Mount Google Drive so the data and trained model are kept between sessions.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
WORK_DIR = Path("/content/drive/MyDrive/assignment03")   # the assignment03 folder you copied to My Drive
PART1_DIR = WORK_DIR / "part1"                            # Part 1 data and the trained U-Net are kept here

missing = [name for name in ["train_images.tif", "train_labels.tif"]
           if not (PART1_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"{missing} not found in My Drive/assignment03/part1. "
                            "Copy the whole assignment03 folder to My Drive (Setup step 1), then rerun this cell.")
print("Part 1 folder:", PART1_DIR)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tifffile
from pathlib import Path
from skimage import segmentation

%matplotlib inline

-----
## Part 1 — Training and image augmentation

Use the three fluorescence images of nuclei and their ground-truth labels provided in `train_images.tif` and `train_labels.tif` (in `My Drive/assignment03/part1/`). Each label image assigns `0` to background and a unique integer to each nucleus.

Train a small U-Net from scratch on the three images without augmentation. Then implement horizontal/vertical flips and the brightness/contrast/gamma transform, and inspect how the trained U-Net predicts on transformed versions of one training image. Keep the image normalization fixed from the original training set; do not normalize each transformed image separately.

In [ ]:
# Load the three training images and their ground-truth labels from your Google Drive folder.
train_images = tifffile.imread(PART1_DIR / "train_images.tif")   # (3, 256, 256), uint16
train_labels = tifffile.imread(PART1_DIR / "train_labels.tif")   # (3, 256, 256), 0 = background, 1..N = nuclei
img_train = train_images[0]

def show(images, titles=None, masks=None, ncols=4, size=3.2, vmin=None, vmax=None):
    n = len(images)
    ncols = min(n, ncols)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * nrows), squeeze=False)
    for k, ax in enumerate(axes.flat):
        ax.axis("off")
        if k >= n:
            continue
        ax.imshow(images[k], cmap="gray", vmin=vmin, vmax=vmax)
        if masks is not None and masks[k] is not None:
            b = segmentation.find_boundaries(masks[k], mode="inner")
            ax.imshow(np.ma.masked_where(~b, b), cmap="autumn", interpolation="none", alpha=0.9)
        if titles is not None:
            ax.set_title(titles[k], fontsize=9)
    plt.tight_layout()
    plt.show()


print("images:", train_images.shape, train_images.dtype, "| labels:", train_labels.shape, train_labels.dtype)
show(list(train_images), [f"training image {k + 1}" for k in range(len(train_images))], ncols=3)
show(list(train_images), [f"training image {k + 1} + labels" for k in range(len(train_images))],
     masks=list(train_labels), ncols=3)

### Exercise 1.1 — Train a U-Net without augmentation

Run the code below to train a small U-Net from randomly initialized weights on the three training images. The target is the foreground of the labels (nucleus vs. background), and training uses no augmentation: no flips, rotations, or intensity changes. The trained weights are saved to `My Drive/assignment03/part1/` and reloaded in Exercise 1.3.

**Question:** Look at the training loss curve. How can you tell from it that the model has learned from the training images?

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("training device:", device)

# Convert instance labels to a binary foreground target (0 = background, 1 = nucleus).
train_foreground = (train_labels > 0).astype(np.int64)

# Use one scale estimated from the original training set for every later input.
intensity_scale = float(np.percentile(train_images, 99.9))
def image_to_tensor(image):
    scaled = np.clip(image.astype(np.float32) / intensity_scale, 0, 1)
    return torch.from_numpy(scaled[None, ...])

x_train = torch.stack([image_to_tensor(image) for image in train_images])
y_train = torch.from_numpy(train_foreground)
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=len(train_images), shuffle=True)


def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )


class SmallUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.enc1 = conv_block(1, 8)
        self.enc2 = conv_block(8, 16)
        self.enc3 = conv_block(16, 32)
        self.middle = conv_block(32, 64)
        self.up3 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec3 = conv_block(64, 32)
        self.up2 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.dec2 = conv_block(32, 16)
        self.up1 = nn.ConvTranspose2d(16, 8, kernel_size=2, stride=2)
        self.dec1 = conv_block(16, 8)
        self.output = nn.Conv2d(8, 2, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        mid = self.middle(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(mid), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.output(d1)


unet = SmallUNet().to(device)  # random initialization: training from scratch
class_weights = torch.tensor([0.35, 3.0], dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(unet.parameters(), lr=1e-3)
n_epochs = 200
loss_history = []

for epoch in range(n_epochs):
    unet.train()
    epoch_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        logits = unet(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    loss_history.append(epoch_loss / len(train_loader))
    if (epoch + 1) % 10 == 0:
        print(f"epoch {epoch + 1:3d}/{n_epochs}: loss={loss_history[-1]:.4f}")


def predict_foreground(image):
    unet.eval()
    with torch.inference_mode():
        logits = unet(image_to_tensor(image)[None, ...].to(device))
        prediction = logits.argmax(dim=1)[0].cpu().numpy()
    return prediction.astype(np.uint8)


fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(np.arange(1, n_epochs + 1), loss_history)
ax.set(xlabel="epoch", ylabel="training loss", title="U-Net trained without augmentation")
plt.show()

# Save a portable PyTorch state-dict checkpoint to Google Drive after training.
unet_checkpoint_path = PART1_DIR / "unet_from_scratch.pth"
torch.save(unet.state_dict(), unet_checkpoint_path)
print("saved U-Net weights to:", unet_checkpoint_path)

**Your answer:**



### Exercise 1.2 — Implement image augmentation

Implement horizontal and vertical flips, and the brightness/contrast/gamma point operation below using NumPy. For an image with `L` intensity levels, the provided formula is

$$s = \operatorname{clip}\!\left(\alpha\left[(L-1)\left(\frac{r}{L-1}
\right)^\gamma - \mu
\right] + \mu + \beta,\;0,\;L-1
\right),$$

where `r` is the input pixel value, `mu` is the mean of the input image, `alpha` controls contrast around that mean, `beta` shifts brightness, and `gamma` changes the intensity curve. Use the dtype's full range for `L`, perform calculations in floating point, clip, and return the input dtype. These functions are used in Exercise 1.3; they are not used in the training loop above.

In [ ]:
def horizontal_flip(img):
    """Return a left-right flipped copy of img."""
    #### YOUR CODE HERE ####
    ...


def vertical_flip(img):
    """Return a top-bottom flipped copy of img."""
    #### YOUR CODE HERE ####
    ...


def brightness_contrast_gamma(img, alpha=1.0, beta=0.0, gamma=1.0):
    """Apply the assignment formula, clip to dtype range, and preserve dtype."""
    #### YOUR CODE HERE ####
    ...


assert np.array_equal(horizontal_flip(horizontal_flip(img_train)), img_train)
assert np.array_equal(vertical_flip(vertical_flip(img_train)), img_train)
assert np.array_equal(brightness_contrast_gamma(img_train), img_train)
assert brightness_contrast_gamma(img_train, alpha=1.0, beta=250, gamma=1.0).dtype == img_train.dtype
assert brightness_contrast_gamma(img_train, gamma=1.5).min() >= 0
assert brightness_contrast_gamma(img_train, gamma=1.5).max() <= np.iinfo(img_train.dtype).max
print("augmentation checks passed")

### Exercise 1.3 — Compare predictions on transformed training images

The U-Net was trained only on the unmodified training images. Run it on the original `img_train`, its horizontal and vertical flips, and a brightness/contrast/gamma-adjusted version. Each flipped prediction is shown on the flipped image it was run on. Masks are overlaid at 50% transparency.

Describe what stays similar and what changes. Compare the original training prediction with its ground-truth label and with the intensity-adjusted prediction. Explain why augmentation during training could help.

In [ ]:
# Reload the saved training checkpoint from Google Drive before inference.
unet_checkpoint_path = PART1_DIR / "unet_from_scratch.pth"
unet = SmallUNet().to(device)
unet.load_state_dict(torch.load(unet_checkpoint_path, map_location=device, weights_only=True))
print("loaded U-Net weights from:", unet_checkpoint_path)

# A stronger intensity shift makes the effect visible; vary these parameters to explore.
img_adjusted = brightness_contrast_gamma(img_train, alpha=1.5, beta=0, gamma=1.5)

inputs = [img_train, horizontal_flip(img_train), vertical_flip(img_train), img_adjusted]
titles = ["original", "horizontal flip", "vertical flip", "adjusted: alpha=1.5, gamma=1.5"]
show(inputs, titles, ncols=4)

pred_original = predict_foreground(img_train)
pred_horizontal = predict_foreground(inputs[1])   # stays in the flipped orientation
pred_vertical = predict_foreground(inputs[2])     # stays in the flipped orientation
pred_adjusted = predict_foreground(img_adjusted)

# Overlay each filled foreground mask on an image: flipped predictions on their flipped images.
predictions = [train_foreground[0], pred_original, pred_horizontal, pred_vertical, pred_adjusted]
backgrounds = [img_train, img_train, inputs[1], inputs[2], img_train]
prediction_titles = ["Ground-truth label", "U-Net: original", "U-Net: horizontal flip",
                     "U-Net: vertical flip", "U-Net: adjusted intensities"]
fig, axes = plt.subplots(1, len(predictions), figsize=(16, 3.5))
for ax, background, mask, title in zip(axes, backgrounds, predictions, prediction_titles):
    ax.imshow(background, cmap="gray")
    ax.imshow(np.ma.masked_where(mask == 0, mask > 0), cmap="autumn", alpha=0.5,
              interpolation="none", vmin=0, vmax=1)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

**Your answer:**



-----
## Part 2 — Human-in-the-loop

In human-in-the-loop training, a person checks a model's predictions, fixes the mistakes, and retrains the model on the corrected labels. You will run the pretrained Cellpose model (**cpsam**), draw the nuclei it missed, fine-tune it on your corrected labels, and check that the fine-tuned model finds them.

Do this part **on your own computer**, using `assignment03/part2/dapi_cutout.tif` in your local copy of the course GitHub repository. Submit two screen captures, `before.png` and `after.png`, through the submission link.

### Steps 1–4 — Do these on your own computer

**Step 1 — Start the Cellpose GUI.** In a terminal with the class environment active, run:

```bash
python -m cellpose
```

**Step 2 — Run the pretrained model.** Choose **File › Load image** and open `assignment03/part2/dapi_cutout.tif`. With **cpsam** selected, click **run**. Take a screenshot with the masks showing and save it as `before.png`.

**Step 3 — Correct the labels.** For each nucleus with no mask, **right-click** on its edge, trace the outline with the mouse, and return to the start point to close the mask. Save with **Ctrl+S** (Mac: **Cmd+S**).

**Step 4 — Fine-tune and run.** Press **Ctrl+T**, or choose **Models › Train new model with image+masks in folder**, and click **OK**. When training is done, select your model in the **user-trained models** panel on the left and click **run**. Take a screenshot of the same area and save it as `after.png`.

### How did the results change?

Compare `before.png` and `after.png` and answer in a few sentences. How do the results of the pretrained and fine-tuned models differ? Did the fine-tuned model find the nuclei that the pretrained model missed? Based on this, what is human-in-the-loop training trying to do?

**Your answer:**

